In [1]:
# ============================================================
# NOTEBOOK 15 — BATCH 4 PREPROCESSING
# CELL 1 — IMPORTS, PATHS, AND MODULE
# ============================================================

from pathlib import Path
import sys

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Project root
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    r"D:\Pancreatic_Cancer_Thesis"
)

DATA_DIR = PROJECT_ROOT / "data"

RAW_CT_DIR = (
    DATA_DIR / "raw_ct"
)

AUTO_LABEL_DIR = (
    DATA_DIR
    / "labels"
    / "Automatic_Labels"
)

MANUAL_LABEL_DIR = (
    DATA_DIR
    / "labels"
    / "Manual_Labels"
)

PROCESSED_DIR = (
    DATA_DIR / "processed"
)

PROCESSED_IMAGES_DIR = (
    PROCESSED_DIR / "images"
)

PROCESSED_MASKS_DIR = (
    PROCESSED_DIR / "masks"
)

BATCH4_ELIGIBILITY_FILE = (
    PROCESSED_DIR
    / "batch4_eligibility.csv"
)

METADATA_FILE = (
    PROCESSED_DIR
    / "metadata.csv"
)

# ------------------------------------------------------------
# IMPORTANT:
# Import the project module as part of the src package.
# Do NOT add PROJECT_ROOT/src to sys.path.
# ------------------------------------------------------------

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import preprocessing as prep


print("=" * 70)
print("NOTEBOOK 15 — BATCH 4 PREPROCESSING")
print("=" * 70)

print("\nProject root:")
print(PROJECT_ROOT)

print("\nPreprocessing module:")
print(prep.__file__)

print("\nBatch 4 eligibility:")
print(BATCH4_ELIGIBILITY_FILE)

print("\nMetadata:")
print(METADATA_FILE)

print("\n✓ Preprocessing module imported successfully.")

NOTEBOOK 15 — BATCH 4 PREPROCESSING

Project root:
D:\Pancreatic_Cancer_Thesis

Preprocessing module:
D:\Pancreatic_Cancer_Thesis\src\preprocessing.py

Batch 4 eligibility:
D:\Pancreatic_Cancer_Thesis\data\processed\batch4_eligibility.csv

Metadata:
D:\Pancreatic_Cancer_Thesis\data\processed\metadata.csv

✓ Preprocessing module imported successfully.


In [2]:
# ============================================================
# CELL 2 — LOAD BATCH 4 ELIGIBILITY
# ============================================================

if not BATCH4_ELIGIBILITY_FILE.exists():
    raise FileNotFoundError(
        f"Batch 4 eligibility file not found:\n"
        f"{BATCH4_ELIGIBILITY_FILE}"
    )

batch4_eligibility = pd.read_csv(
    BATCH4_ELIGIBILITY_FILE
)

print("=" * 70)
print("BATCH 4 ELIGIBILITY")
print("=" * 70)

print(
    "Total rows :",
    len(batch4_eligibility)
)

print(
    "Eligible   :",
    int(batch4_eligibility["eligible"].sum())
)

print(
    "Excluded   :",
    int(
        (~batch4_eligibility["eligible"]).sum()
    )
)

assert len(batch4_eligibility) == 535
assert batch4_eligibility["eligible"].sum() == 535
assert (~batch4_eligibility["eligible"]).sum() == 0

print("\n✓ All 535 Batch 4 cases are eligible.")

BATCH 4 ELIGIBILITY
Total rows : 535
Eligible   : 535
Excluded   : 0

✓ All 535 Batch 4 cases are eligible.


In [3]:
# ============================================================
# CELL 3 — BUILD BATCH 4 STUDY ID LIST
# ============================================================

batch4_eligible = (
    batch4_eligibility[
        batch4_eligibility["eligible"]
    ]
    .copy()
)

batch4_eligible_ids = (
    batch4_eligible["study_id"]
    .astype(str)
    .tolist()
)

print("=" * 70)
print("BATCH 4 STUDY IDS")
print("=" * 70)

print(
    "Eligible cases:",
    len(batch4_eligible_ids)
)

print("\nFirst 10:")
for study_id in batch4_eligible_ids[:10]:
    print(" ", study_id)

print("\nLast 10:")
for study_id in batch4_eligible_ids[-10:]:
    print(" ", study_id)

assert len(batch4_eligible_ids) == 535
assert len(set(batch4_eligible_ids)) == 535

print("\n✓ Study-ID list verified.")

BATCH 4 STUDY IDS
Eligible cases: 535

First 10:
  101691_00001
  101692_00001
  101693_00001
  101694_00001
  101695_00001
  101696_00001
  101697_00001
  101698_00001
  101699_00001
  101700_00001

Last 10:
  102214_00001
  102215_00001
  102216_00001
  102217_00001
  102218_00001
  102219_00001
  102220_00001
  102221_00001
  102222_00001
  102223_00001

✓ Study-ID list verified.


In [4]:
# ============================================================
# CELL 4 — CHECK EXISTING BATCH 4 OUTPUTS
# ============================================================

PROCESSED_IMAGES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PROCESSED_MASKS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

existing_images = {
    p.stem
    for p in PROCESSED_IMAGES_DIR.glob("*.npy")
}

existing_masks = {
    p.stem
    for p in PROCESSED_MASKS_DIR.glob("*.npy")
}

batch4_set = set(
    batch4_eligible_ids
)

batch4_existing_images = (
    batch4_set & existing_images
)

batch4_existing_masks = (
    batch4_set & existing_masks
)

batch4_missing_images = (
    batch4_set - existing_images
)

batch4_missing_masks = (
    batch4_set - existing_masks
)

print("=" * 70)
print("EXISTING BATCH 4 OUTPUTS")
print("=" * 70)

print(
    "Batch 4 eligible cases :",
    len(batch4_set)
)

print(
    "Existing images        :",
    len(batch4_existing_images)
)

print(
    "Existing masks         :",
    len(batch4_existing_masks)
)

print(
    "Missing images         :",
    len(batch4_missing_images)
)

print(
    "Missing masks          :",
    len(batch4_missing_masks)
)

print("\nTotal processed images:",
      len(existing_images))

print("Total processed masks :",
      len(existing_masks))

EXISTING BATCH 4 OUTPUTS
Batch 4 eligible cases : 535
Existing images        : 0
Existing masks         : 0
Missing images         : 535
Missing masks          : 535

Total processed images: 1703
Total processed masks : 1703


In [5]:
# ============================================================
# CELL 5 — VERIFY PREPROCESSING CONFIGURATION
# ============================================================

print("=" * 70)
print("PREPROCESSING CONFIGURATION")
print("=" * 70)

config_names = [
    "DEFAULT_TARGET_SPACING",
    "DEFAULT_ROI_SIZE",
    "DEFAULT_HU_WINDOW",
]

for name in config_names:

    if hasattr(prep, name):

        print(
            f"{name:<25}:",
            getattr(prep, name)
        )

    else:

        print(
            f"{name:<25}: NOT FOUND"
        )

print("\nExpected final representation:")
print("  Image shape : (128, 160, 192)")
print("  Image dtype : float32")
print("  Image range : [0, 1]")
print("  Mask shape  : (128, 160, 192)")
print("  Mask dtype  : uint8")

PREPROCESSING CONFIGURATION
DEFAULT_TARGET_SPACING   : (1.0, 1.0, 3.0)
DEFAULT_ROI_SIZE         : (128, 160, 192)
DEFAULT_HU_WINDOW        : (-150, 250)

Expected final representation:
  Image shape : (128, 160, 192)
  Image dtype : float32
  Image range : [0, 1]
  Mask shape  : (128, 160, 192)
  Mask dtype  : uint8


In [6]:
# ============================================================
# CELL 6 — PROCESS BATCH 4
# ============================================================

print("=" * 70)
print("STARTING BATCH 4 PREPROCESSING")
print("=" * 70)

print(
    "Requested cases:",
    len(batch4_eligible_ids)
)

print(
    "Overwrite:",
    False
)

print(
    "\nExisting processed outputs will be preserved."
)

batch4_metadata = prep.process_dataset(
    study_ids=batch4_eligible_ids,
    overwrite=False,
)

print("\n" + "=" * 70)
print("BATCH 4 PREPROCESSING COMPLETE")
print("=" * 70)

if batch4_metadata is not None:

    print(
        "Metadata rows returned:",
        len(batch4_metadata)
    )

STARTING BATCH 4 PREPROCESSING
Requested cases: 535
Overwrite: False

Existing processed outputs will be preserved.


Processing dataset:   0%|          | 0/535 [00:00<?, ?it/s]

After Resample   : -1114 2250
After Clip       : -150 250
After Normalize  : 0.0 1.0
After Crop       : 0.0 1.0
After Pad        : 0.0 1.0
Final            : 0.0 1.0
After Resample   : -1024 3071
After Clip       : -150 250
After Normalize  : 0.0 1.0
After Crop       : 0.0 1.0
After Pad        : 0.0 1.0
Final            : 0.0 1.0
After Resample   : -1024 1872
After Clip       : -150 250
After Normalize  : 0.0 1.0
After Crop       : 0.0 1.0
After Pad        : 0.0 1.0
Final            : 0.0 1.0
After Resample   : -1024 1914
After Clip       : -150 250
After Normalize  : 0.0 1.0
After Crop       : 0.0 1.0
After Pad        : 0.0 1.0
Final            : 0.0 1.0
After Resample   : -1024 3071
After Clip       : -150 250
After Normalize  : 0.0 1.0
After Crop       : 0.0 1.0
After Pad        : 0.0 1.0
Final            : 0.0 1.0
After Resample   : -1024 1348
After Clip       : -150 250
After Normalize  : 0.0 1.0
After Crop       : 0.0 1.0
After Pad        : 0.0 1.0
Final            : 0.0 1.0
Afte

In [4]:
# ============================================================
# FIND THE ONE BATCH 4 CASE MISSING FROM METADATA
# ============================================================

print("=" * 70)
print("BATCH 4 COMPLETION AUDIT")
print("=" * 70)

# Reload the eligibility list to make this independent
# of notebook state.

batch4_eligibility_check = pd.read_csv(
    BATCH4_ELIGIBILITY_FILE
)

batch4_ids = set(
    batch4_eligibility_check[
        batch4_eligibility_check["eligible"]
    ]["study_id"]
    .astype(str)
)

# Current metadata
metadata_check = pd.read_csv(
    METADATA_FILE
)

metadata_ids = set(
    metadata_check["study_id"]
    .astype(str)
)

# ------------------------------------------------------------
# Cases missing from metadata
# ------------------------------------------------------------

missing_from_metadata = sorted(
    batch4_ids - metadata_ids
)

# ------------------------------------------------------------
# Cases in metadata but not Batch 4
# ------------------------------------------------------------

unexpected_metadata_cases = sorted(
    metadata_ids - (
        set(
            pd.read_csv(
                BATCH4_ELIGIBILITY_FILE
            )["study_id"]
            .astype(str)
        )
    )
)

print(
    "Batch 4 eligible cases :",
    len(batch4_ids)
)

print(
    "Current metadata rows  :",
    len(metadata_check)
)

print(
    "Missing from metadata  :",
    len(missing_from_metadata)
)

print("\nMissing Batch 4 case(s):")

for study_id in missing_from_metadata:
    print(" ", study_id)

print("\n" + "-" * 70)

# ------------------------------------------------------------
# Check actual image/mask files for the missing case
# ------------------------------------------------------------

for study_id in missing_from_metadata:

    image_path = (
        PROCESSED_IMAGES_DIR
        / f"{study_id}.npy"
    )

    mask_path = (
        PROCESSED_MASKS_DIR
        / f"{study_id}.npy"
    )

    print("\nCASE:", study_id)

    print(
        "Image exists:",
        image_path.exists()
    )

    print(
        "Mask exists :",
        mask_path.exists()
    )

    if image_path.exists():

        image = np.load(
            image_path,
            mmap_mode="r"
        )

        print(
            "Image shape:",
            image.shape
        )

        print(
            "Image dtype:",
            image.dtype
        )

    if mask_path.exists():

        mask = np.load(
            mask_path,
            mmap_mode="r"
        )

        print(
            "Mask shape:",
            mask.shape
        )

        print(
            "Mask dtype:",
            mask.dtype
        )

        print(
            "Mask labels:",
            np.unique(mask)
        )

BATCH 4 COMPLETION AUDIT
Batch 4 eligible cases : 535
Current metadata rows  : 2237
Missing from metadata  : 1

Missing Batch 4 case(s):
  102133_00001

----------------------------------------------------------------------

CASE: 102133_00001
Image exists: False
Mask exists : False


In [5]:
# ============================================================
# DIAGNOSE 102133_00001
# ============================================================

study_id = "102133_00001"

print("=" * 70)
print("DIAGNOSING BATCH 4 CASE")
print("=" * 70)

ct_path = (
    RAW_CT_DIR
    / f"{study_id}_0000.nii.gz"
)

automatic_label_path = (
    AUTO_LABEL_DIR
    / f"{study_id}.nii.gz"
)

manual_label_path = (
    MANUAL_LABEL_DIR
    / f"{study_id}.nii.gz"
)

print("\nCT:")
print(ct_path)
print("Exists:", ct_path.exists())

print("\nAutomatic label:")
print(automatic_label_path)
print("Exists:", automatic_label_path.exists())

print("\nManual label:")
print(manual_label_path)
print("Exists:", manual_label_path.exists())


# ------------------------------------------------------------
# Which label did eligibility select?
# ------------------------------------------------------------

row = batch4_eligibility[
    batch4_eligibility["study_id"].astype(str)
    == study_id
]

if len(row) != 1:
    raise RuntimeError(
        f"Expected exactly one eligibility row, found {len(row)}"
    )

selected_label_type = (
    row.iloc[0]["selected_label_type"]
)

print("\nSelected label type:")
print(selected_label_type)

if selected_label_type == "automatic":
    selected_label_path = automatic_label_path
elif selected_label_type == "manual":
    selected_label_path = manual_label_path
else:
    selected_label_path = None

print("\nSelected label:")
print(selected_label_path)
print(
    "Exists:",
    selected_label_path.exists()
    if selected_label_path
    else False
)


# ------------------------------------------------------------
# Try loading CT
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("CT READ TEST")
print("-" * 70)

try:

    ct_image = sitk.ReadImage(
        str(ct_path)
    )

    print("✓ CT readable")
    print("Size      :", ct_image.GetSize())
    print("Spacing   :", ct_image.GetSpacing())
    print("Origin    :", ct_image.GetOrigin())

except Exception as e:

    print("✗ CT failed to load")
    print("Error:", repr(e))


# ------------------------------------------------------------
# Try loading label
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("LABEL READ TEST")
print("-" * 70)

try:

    label_image = sitk.ReadImage(
        str(selected_label_path)
    )

    print("✓ Label readable")
    print("Size      :", label_image.GetSize())
    print("Spacing   :", label_image.GetSpacing())
    print("Origin    :", label_image.GetOrigin())

except Exception as e:

    print("✗ Label failed to load")
    print("Error:", repr(e))


print("\n" + "=" * 70)
print("DIAGNOSTIC COMPLETE")
print("=" * 70)

DIAGNOSING BATCH 4 CASE

CT:
D:\Pancreatic_Cancer_Thesis\data\raw_ct\102133_00001_0000.nii.gz
Exists: True

Automatic label:
D:\Pancreatic_Cancer_Thesis\data\labels\Automatic_Labels\102133_00001.nii.gz
Exists: True

Manual label:
D:\Pancreatic_Cancer_Thesis\data\labels\Manual_Labels\102133_00001.nii.gz
Exists: False

Selected label type:
automatic

Selected label:
D:\Pancreatic_Cancer_Thesis\data\labels\Automatic_Labels\102133_00001.nii.gz
Exists: True

----------------------------------------------------------------------
CT READ TEST
----------------------------------------------------------------------
✗ CT failed to load
Error: NameError("name 'sitk' is not defined")

----------------------------------------------------------------------
LABEL READ TEST
----------------------------------------------------------------------
✗ Label failed to load
Error: NameError("name 'sitk' is not defined")

DIAGNOSTIC COMPLETE


In [6]:
# ============================================================
# DIAGNOSE 102133_00001 — CORRECTED
# ============================================================

import SimpleITK as sitk

study_id = "102133_00001"

print("=" * 70)
print("DIAGNOSING BATCH 4 CASE")
print("=" * 70)

ct_path = (
    RAW_CT_DIR
    / f"{study_id}_0000.nii.gz"
)

automatic_label_path = (
    AUTO_LABEL_DIR
    / f"{study_id}.nii.gz"
)

manual_label_path = (
    MANUAL_LABEL_DIR
    / f"{study_id}.nii.gz"
)

print("\nCT:")
print(ct_path)
print("Exists:", ct_path.exists())

print("\nAutomatic label:")
print(automatic_label_path)
print("Exists:", automatic_label_path.exists())

print("\nManual label:")
print(manual_label_path)
print("Exists:", manual_label_path.exists())


# ------------------------------------------------------------
# Check eligibility selection
# ------------------------------------------------------------

row = batch4_eligibility[
    batch4_eligibility["study_id"].astype(str)
    == study_id
]

if len(row) != 1:
    raise RuntimeError(
        f"Expected exactly one eligibility row, found {len(row)}"
    )

selected_label_type = (
    row.iloc[0]["selected_label_type"]
)

print("\nSelected label type:")
print(selected_label_type)

if selected_label_type == "automatic":
    selected_label_path = automatic_label_path

elif selected_label_type == "manual":
    selected_label_path = manual_label_path

else:
    raise RuntimeError(
        f"Invalid selected label type: {selected_label_type}"
    )

print("\nSelected label:")
print(selected_label_path)
print("Exists:", selected_label_path.exists())


# ------------------------------------------------------------
# CT readability
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("CT READ TEST")
print("-" * 70)

try:

    ct_image = sitk.ReadImage(
        str(ct_path)
    )

    print("✓ CT readable")
    print("Size      :", ct_image.GetSize())
    print("Spacing   :", ct_image.GetSpacing())
    print("Origin    :", ct_image.GetOrigin())
    print("Direction :", ct_image.GetDirection())

except Exception as e:

    print("✗ CT failed to load")
    print("Error:", repr(e))
    raise


# ------------------------------------------------------------
# Label readability
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("LABEL READ TEST")
print("-" * 70)

try:

    label_image = sitk.ReadImage(
        str(selected_label_path)
    )

    print("✓ Label readable")
    print("Size      :", label_image.GetSize())
    print("Spacing   :", label_image.GetSpacing())
    print("Origin    :", label_image.GetOrigin())
    print("Direction :", label_image.GetDirection())

except Exception as e:

    print("✗ Label failed to load")
    print("Error:", repr(e))
    raise


# ------------------------------------------------------------
# Geometry
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("GEOMETRY CHECK")
print("-" * 70)

shape_match = (
    ct_image.GetSize()
    == label_image.GetSize()
)

spacing_match = np.allclose(
    ct_image.GetSpacing(),
    label_image.GetSpacing(),
    atol=1e-4,
)

origin_match = np.allclose(
    ct_image.GetOrigin(),
    label_image.GetOrigin(),
    atol=1e-4,
)

direction_match = np.allclose(
    ct_image.GetDirection(),
    label_image.GetDirection(),
    atol=1e-5,
)

print("Shape match    :", shape_match)
print("Spacing match  :", spacing_match)
print("Origin match   :", origin_match)
print("Direction match:", direction_match)

print("\n" + "=" * 70)
print("DIAGNOSTIC COMPLETE")
print("=" * 70)

DIAGNOSING BATCH 4 CASE

CT:
D:\Pancreatic_Cancer_Thesis\data\raw_ct\102133_00001_0000.nii.gz
Exists: True

Automatic label:
D:\Pancreatic_Cancer_Thesis\data\labels\Automatic_Labels\102133_00001.nii.gz
Exists: True

Manual label:
D:\Pancreatic_Cancer_Thesis\data\labels\Manual_Labels\102133_00001.nii.gz
Exists: False

Selected label type:
automatic

Selected label:
D:\Pancreatic_Cancer_Thesis\data\labels\Automatic_Labels\102133_00001.nii.gz
Exists: True

----------------------------------------------------------------------
CT READ TEST
----------------------------------------------------------------------
✓ CT readable
Size      : (1024, 1024, 350)
Spacing   : (0.3910059928894043, 0.3910059928894043, 2.0)
Origin    : (-172.6560516357422, -199.9998016357422, 1007.5)
Direction : (1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)

----------------------------------------------------------------------
LABEL READ TEST
----------------------------------------------------------------------
✓ Label

In [12]:
# ============================================================
# MEMORY CHECK BEFORE 102133_00001 RETRY
# ============================================================

import gc
import os
import psutil
import SimpleITK as sitk

process = psutil.Process(os.getpid())

print("=" * 70)
print("MEMORY CHECK")
print("=" * 70)

print(
    "RAM currently used:",
    process.memory_info().rss / (1024 ** 3),
    "GB"
)

print(
    "Available RAM:",
    psutil.virtual_memory().available / (1024 ** 3),
    "GB"
)

print("\n102133_00001 mask:")
mask_path = (
    AUTO_LABEL_DIR
    / "102133_00001.nii.gz"
)

print("Path:", mask_path)
print("File size:", mask_path.stat().st_size / (1024 ** 2), "MB")

print("\nReleasing unneeded Python objects...")

gc.collect()

print(
    "RAM after gc:",
    process.memory_info().rss / (1024 ** 3),
    "GB"
)

print(
    "Available RAM after gc:",
    psutil.virtual_memory().available / (1024 ** 3),
    "GB"
)

MEMORY CHECK
RAM currently used: 0.0251617431640625 GB
Available RAM: 1.7475814819335938 GB

102133_00001 mask:
Path: D:\Pancreatic_Cancer_Thesis\data\labels\Automatic_Labels\102133_00001.nii.gz
File size: 2.801799774169922 MB

Releasing unneeded Python objects...
RAM after gc: 0.1019287109375 GB
Available RAM after gc: 1.6706466674804688 GB


In [2]:
# ============================================================
# FRESH-KERNEL ONE-CASE MEMORY TEST
# ============================================================

import gc
import os
import psutil
import numpy as np
import SimpleITK as sitk

study_id = "102133_00001"

process = psutil.Process(os.getpid())

print("=" * 70)
print("FRESH-KERNEL MEMORY TEST")
print("=" * 70)

print(
    "Available RAM:",
    psutil.virtual_memory().available / (1024 ** 3),
    "GB"
)

# ------------------------------------------------------------
# Load CT only
# ------------------------------------------------------------

ct_path = (
    RAW_CT_DIR
    / f"{study_id}_0000.nii.gz"
)

print("\nLoading CT...")

ct = sitk.ReadImage(
    str(ct_path)
)

print("✓ CT loaded")
print("Size:", ct.GetSize())
print("Spacing:", ct.GetSpacing())

print(
    "Available RAM after CT:",
    psutil.virtual_memory().available / (1024 ** 3),
    "GB"
)

# ------------------------------------------------------------
# Release CT
# ------------------------------------------------------------

del ct
gc.collect()

print(
    "\nAvailable RAM after releasing CT:",
    psutil.virtual_memory().available / (1024 ** 3),
    "GB"
)

# ------------------------------------------------------------
# Load automatic mask only
# ------------------------------------------------------------

mask_path = (
    AUTO_LABEL_DIR
    / f"{study_id}.nii.gz"
)

print("\nLoading automatic mask...")

mask = sitk.ReadImage(
    str(mask_path)
)

print("✓ Mask loaded")
print("Size:", mask.GetSize())
print("Spacing:", mask.GetSpacing())

print(
    "Available RAM after mask:",
    psutil.virtual_memory().available / (1024 ** 3),
    "GB"
)

del mask
gc.collect()

print(
    "\nAvailable RAM after releasing mask:",
    psutil.virtual_memory().available / (1024 ** 3),
    "GB"
)

FRESH-KERNEL MEMORY TEST
Available RAM: 3.0389404296875 GB

Loading CT...
✓ CT loaded
Size: (1024, 1024, 350)
Spacing: (0.3910059928894043, 0.3910059928894043, 2.0)
Available RAM after CT: 2.4329795837402344 GB

Available RAM after releasing CT: 3.1127471923828125 GB

Loading automatic mask...
✓ Mask loaded
Size: (1024, 1024, 350)
Spacing: (0.3910059928894043, 0.3910059928894043, 2.0)
Available RAM after mask: 2.4428672790527344 GB

Available RAM after releasing mask: 4.132396697998047 GB


In [3]:
# ============================================================
# RETRY 102133_00001 IN FRESH KERNEL
# ============================================================

import gc

study_id = "102133_00001"

print("=" * 70)
print("PROCESSING 102133_00001 — FRESH KERNEL")
print("=" * 70)

gc.collect()

retry_metadata = prep.process_dataset(
    study_ids=[study_id],
    overwrite=False,
)

print("\n" + "=" * 70)
print("RETRY COMPLETE")
print("=" * 70)

if retry_metadata is not None:
    print(
        "Metadata rows returned:",
        len(retry_metadata)
    )

PROCESSING 102133_00001 — FRESH KERNEL


Processing dataset:   0%|          | 0/1 [00:00<?, ?it/s]

After Resample   : -2048 3822
After Clip       : -150 250
After Normalize  : 0.0 1.0
After Crop       : 0.0 1.0
After Pad        : 0.0 1.0
Final            : 0.0 1.0

RETRY COMPLETE
Metadata rows returned: 2238


In [5]:
# ============================================================
# FINAL 2,238-CASE DATASET VERIFICATION
# ============================================================

print("=" * 70)
print("FINAL 2,238-CASE DATASET VERIFICATION")
print("=" * 70)

summary, report = prep.verify_dataset()

print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

for key, value in summary.items():
    print(f"{key:<25}: {value}")

FINAL 2,238-CASE DATASET VERIFICATION


Verifying dataset:   0%|          | 0/2238 [00:00<?, ?it/s]

Processed Dataset Verification
num_cases                : 2238
missing_images           : 0
missing_masks            : 0
invalid_image_shape      : 0
invalid_mask_shape       : 0
invalid_image_dtype      : 0
invalid_mask_dtype       : 0
invalid_image_range      : 0
invalid_mask_labels      : 0
duplicate_study_ids      : 0

✓ All processed cases passed verification.

FINAL SUMMARY
num_cases                : 2238
missing_images           : 0
missing_masks            : 0
invalid_image_shape      : 0
invalid_mask_shape       : 0
invalid_image_dtype      : 0
invalid_mask_dtype       : 0
invalid_image_range      : 0
invalid_mask_labels      : 0
duplicate_study_ids      : 0
